# 02 · SVS の設計意図

**改造する人向け**のノートです。各段が何をしていて、**なぜ**そうしているかを説明します。

```
note + lyric
   └─① duration 計算 ─② duration postprocess ─③ F0（PrettyPitch）─④ F0 postprocess
                                                  └─⑤ 音響（LeapSinger）─⑥ ボコーダ（NHVSing）
```

## 段階的なパイプラインになっている理由

一体型（楽譜→波形を1モデル）にしない理由は **人が触れる中間表現を残すため**です。
持続長・F0 は人間が修正することができ、実際 多くの歌声合成用のソフトウェアはそうなっています。
分けておけば、F0 だけ差し替える・持続長だけ規則で直す、といった改造ができます。

代償は段ごとの誤差の蓄積ですが、各段の入出力の仕様を固定しておけば影響を局所化できます。

## 段間の受け渡し

| 境界 | 受け渡すもの |
|---|---|
| ②→③ | `PhonemeSequence`（`phonemes` / `dur_sec` / `note_midi` / `is_vowel` / `note_event_starts`） |
| ③→④→⑤ | `f0_hz [T]` ＋ `uv [T]`（1 フレーム = 256/44100 秒 = 172.266 fps） |
| ⑤→⑥ | `mel [128, T]`（自然対数）＋ 同じ `f0` / `uv`。mel にはこの境界で音量合わせとリミッタが掛かります（§3） |

**フレーム格子が全段で同一**（44.1 kHz / hop 256）なのが効いていて、
リサンプルも補間も挟まずに繋がります。

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from svs.score_io import load_score
from svs.duration_postprocess import (Note, Phrase, postprocess_durations, to_frames,
                                      split_syllable, _syllabify, is_nucleus_ph)
from duration_model.profile import default_profile, load_profile, VOWEL_FLOOR_SEC
from svs.plot import plot_alignment, plot_f0, plot_mel, mel_from_wav

prof = default_profile()                    # 以降のデモセルはこちらを使う
ship = load_profile("cons_dur_3singer")     # 合成が実際に使うプロファイル

print("組込み既定（プロファイル JSON もレシピも無いときのフォールバック）")
print(f"  母音 floor     : {prof.v_min()*1000:.0f} ms   (VOWEL_FLOOR_SEC)")
print(f"  子音 floor     : {prof.c_min('k')*1000:.0f} ms   (全子音一律)")
print(f"  先頭子音 floor : {prof.lead_c_min('k')*1000:.0f} ms   (指定が無いので子音 floor へ落ちる)")

print("\nduration_model/profiles/cons_dur_3singer.json（配布ウェイトの対）")
print(f"  母音 floor     : {ship.v_min()*1000:.0f} ms")
print(f"  子音 floor     : {len(ship.cons_floor)} 子音   実測長の 平均 − min(σ, 0.6·平均)")
print(f"  先頭子音 floor : {len(ship.lead_cons_floor)} 子音   フレーズ頭での実測 p75")
for c in ("k", "s", "n", "t"):
    print(f"      {c}: 中間 {ship.c_min(c)*1000:3.0f} ms / 先頭 {ship.lead_c_min(c)*1000:3.0f} ms")

## 1. duration に掛ける補正

生の「ノート長 ÷ 音素数」では歌になりません。子音の長さは**子音長モデル**
（`duration_model/`、score-free 学習）が 1 子音ずつ予測し、そこに floor と圧縮が掛かります。

| 補正 | 何をするか | なぜ |
|---|---|---|
| **子音長モデル** | 話者・スタイル・前後音素・ノート長から自然長を予測 | 子音の長さは音素ごとにもテンポごとにも違う。固定値では表現できない |
| **onset 前送り** | 子音をノートの**前**に置く | 人は子音をビートの前に出して、母音をビートに合わせて歌う |
| **overlap による母音削り** | 次ノートの onset が食い込む分、前の母音を短くする | 子音の時間をどこかから取る必要がある。母音の末尾が自然 |
| **子音 floor** `c_min` | 子音が潰れすぎない下限（子音別。実測長の平均 − σ、σ は平均の 0.6 倍で頭打ち） | 早口でも子音には物理的な最短長がある。潰すと発音が消える |
| **先頭子音 floor** `lead_c_min` | フレーズ先頭の onset だけ別の下限（フレーズ頭での実測 p75） | 歌い出しの子音は中間位置と長さの分布が違う。多くは長いが、破裂音は先行する閉鎖が pau 側に付くので逆に短い |
| **母音 floor** 75 ms | 母音がこれ以下に潰れない | 母音が 0 になると音そのものが消える |

ノート区間に floor が収まるときは floor をそのまま保証します（子音は自然長、母音が残りを取る）。
**母音 floor + 子音 floor がノート長に収まらないときだけ**、区間を音素ごとの重み比で
按分します（`_compress_region`）。重み `phoneme_weight` は既定 1.0 で、既定のままなら
その区間に居る子音と母音が同じ長さになります。子音の合計を固定割合で頭打ちにすると、
子音が並ぶ区間だけ 1 本あたりが潰れるためです。
floor の分位点・母音 floor・音素ごとの重みはレシピの `duration:` で DB ごとに決め、
`python -m duration_model.build_profile` が `duration_model/profiles/<ckpt名>.json` に焼きます。

`svs.render` は既定でモデルを使い、`--cons_dur_ckpt none` で固定長（80 ms）に
フォールバックします。以下のセルはモデル無し（固定長）で floor の効きだけを見ます。

### 多モーラノートの扱い

1 ノートに複数モーラが入る譜面（例「かか」を 1 ノート）は、
**ノートを `_syllabify` のグループ数で等分**してから通常経路に流します
（`split_phrase_syllables`）。
つまり「3 モーラ 1 ノート」は「1 モーラ 1 ノート × 3」と**完全に同じ出力**になります。
特別扱いのコードを増やさずに済むのが狙いです。


In [ ]:
# グループへの分割（境界は核の数で決まる）。核を持たないグループもあるので「音節」とは呼ばない。
# cl は核ではないので、後ろに核があればその onset として**次**のグループに付く。
for phs in (["k","a"], ["k","a","k","a"], ["a","k","a"], ["i","cl"], ["cl"], ["k","s","t"]):
    groups = _syllabify(phs)
    print(f'{str(phs):22s} -> {len(groups)} グループ {groups}   split_syllable={split_syllable(phs)}')

In [ ]:
def one_note(phs, dur, midi=62):
    return postprocess_durations(
        Phrase(notes=[Note(phonemes=list(phs), pitch_midi=midi, dur_sec=dur)],
               leading_pau_sec=0.1, trailing_pau_sec=0.1), rest_before=1e9)

# 「N モーラ 1 ノート」== 「1 モーラ × N ノート」
a = one_note(["k","a","k","a"], 0.6)
b = postprocess_durations(Phrase(notes=[Note(phonemes=["k","a"], pitch_midi=62, dur_sec=0.3),
                                        Note(phonemes=["k","a"], pitch_midi=62, dur_sec=0.3)],
                                 leading_pau_sec=0.1, trailing_pau_sec=0.1), rest_before=1e9)
print("かか(1ノート0.6s) == か(0.3s)+か(0.3s):",
      list(a.phonemes) == list(b.phonemes) and np.array_equal(a.dur_sec, b.dur_sec))
print("  ", " ".join(f"{p}:{d:.3f}" for p, d in zip(a.phonemes, a.dur_sec)))

### floor の効きを見る

同じノートを 0.60 → 0.24 → 0.12 秒と縮めます。
母音だけが縮み、子音は floor で踏みとどまるのが見えます。

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 6))
for ax, dur in zip(axes, (0.60, 0.24, 0.12)):
    s = one_note(["k","a"], dur)
    plot_alignment(s, ax=ax)
    cons = float(s.dur_sec[list(s.phonemes).index("k")])
    vow = float(s.dur_sec[list(s.phonemes).index("a")])
    ax.set_title(f"note {dur*1000:.0f} ms  ->  k {cons*1000:.0f} ms / a {vow*1000:.0f} ms "
                 f"(vowel floor {prof.vowel_floor*1000:.0f} ms)")
plt.tight_layout(); plt.show()

## 2. F0 に掛ける補正

F0 は `F0 = 2^(baseline + plain + vibrato)` で組み立てます。

### 2-1. baseline（σ20 ms）

楽譜の音高を階段状に並べ、ガウシアンで平滑したものです。

**なぜ σ を大きくしない（例えば 50 ms に）のか。** baseline を滑らかにしすぎると、
ノート間の移行（ポルタメント）を baseline が吸ってしまい、モデルが学ぶべき
「人がどう繋いだか」が教師から消えます。σ20 ms は「階段の角を取る」程度に留め、
繋ぎ方は plain チャンネルに学ばせる、という分担です。

### 2-2. Stage-1：Blaauw の定数オフセット補正（σ30 ms 平滑）

ノートごとに「知覚される音高」を重み付き平均で推定し、楽譜の音高との**差を定数で**引きます。

**なぜ定数か。** 曲線の形を変えずに平行移動するので、**しゃくり・落ち・ビブラートの形が保たれます**。
フレーム単位で目標へ引っ張ると、歌唱表現ごと潰れて機械的になります。

重みは Blaauw & Bonada (2017) に沿って 3 つの積: 端を減衰させる Tukey 窓 ×
安定度（Savitzky-Golay の傾きが小さいほど重い）× 目標音高への近さ。
「人は音の真ん中の安定した部分でピッチを判断する」という知覚モデルです。

### 2-3. Stage-2：非対称 Tukey の引き寄せ

母音フレームを、窓で重み付けしながら目標音高へ `alpha` の割合だけ引き寄せます。

**この SVS 経路では `alpha=0.8` / `lead_taper=0.6` / `trail_taper=0.1`** を使っています。
対称 Hann の 0.5 より**意図的に強い**設定です。非対称なのは、**入口のしゃくりは残したい・終わり際の
オーバーシュートは抑えたい**という非対称な要求があるからです。
左を長くテーパ（0.6）すると入口が緩く、右を短く（0.1）すると終端がしっかり引かれます。

> 補正の実装は `prettypitch/correction.py` の 1 箇所だけです。

### 2-4. 補正しないもの

* **休符**: `pitch == 0` のノートは飛ばす（無声なので意味がない）
* **vibrato**: 補正はすべて plain に掛かり、vibrato は最後に掛け戻す
* **子音**: `cons_weight=0` が既定。子音区間のピッチは知覚に寄与しないので重み 0

In [ ]:
from svs.render import load_models, _f0_score_from_seq

ctx = load_models(spk_id=2, device="cpu")
phr = load_score(ROOT / "examples" / "kirakiraboshi.ust")[0]
seq = postprocess_durations(phr, rest_before=1e9)
score = _f0_score_from_seq(seq, spk_id=2, style_id=0)
fr = ctx.frame_rate

def synth(**kw):
    return ctx.pp.synthesize_f0(ctx.f0_model, ctx.f0_cfg, score, device="cpu", seed=0, **kw)

raw   = synth(correction=False, hann_alpha=0.0)   # 補正なし
st1   = synth(correction=True,  hann_alpha=0.0)   # Stage-1 のみ
both  = synth()                                   # 既定（Stage-1 + Stage-2 alpha=0.8）

t = np.arange(len(both["f0_hz"])) / fr
tgt = 69 + 12*(both["baseline_log2"] - np.log2(440))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(t, tgt, "--", color="#888", lw=1.0, label="baseline (target)")
for o, c_, lab in ((raw, "#c62828", "no correction"),
                   (st1, "#ef6c00", "Stage-1 only"),
                   (both, "#1565c0", "Stage-1 + Stage-2 (default)")):
    m = np.where(o["uv"] > 0, 69 + 12*np.log2(np.maximum(o["f0_hz"], 1e-6)/440), np.nan)
    axes[0].plot(t, m, lw=1.1, color=c_, label=lab)
axes[0].set_ylabel("pitch [MIDI]"); axes[0].legend(fontsize=8); axes[0].set_title("2-stage pitch correction")
for o, c_, lab in ((raw, "#c62828", "no correction"),
                   (st1, "#ef6c00", "Stage-1 only"),
                   (both, "#1565c0", "default")):
    m = np.where(o["uv"] > 0, 69 + 12*np.log2(np.maximum(o["f0_hz"], 1e-6)/440), np.nan)
    axes[1].plot(t, (m - tgt)*100, lw=1.0, color=c_, label=lab)
axes[1].axhline(0, color="#888", lw=0.6)
axes[1].set_ylabel("deviation from target [cent]"); axes[1].set_xlabel("time [s]"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

for o, lab in ((raw, "no correction"), (st1, "Stage-1 only"), (both, "default")):
    v = o["uv"] > 0
    m = 69 + 12*np.log2(np.maximum(o["f0_hz"], 1e-6)/440)
    print(f"{lab:22s} |F0 - target| 中央値 = {np.median(np.abs(m[v]-tgt[v]))*100:6.1f} cent")
print()
print("vibrato は補正の影響を受けない:",
      np.array_equal(raw["vib_delta"], both["vib_delta"]))

## 3. mel に掛ける補正

音響モデルが出した mel には、ボコーダへ渡す前に **2 つだけ**手が入ります。どちらも
`svs/render.py` の `_vocode` の中、この 1 か所だけです。

| 補正 | 何をするか | なぜ |
|---|---|---|
| **ラウドネス合わせ** `_mel_loudness_offset` | 有声フレームの mel の大きさが目標になるよう、ln-mel 全体に定数 δ を足す | ボコーダは学習データを有声区間の実効値 0.111 に正規化して学習している。その音量で駆動するのがいちばん得意。波形側で上げると、信号と一緒にボコーダのノイズ床も上がる |
| **フレームリミッタ** `_mel_frame_limit` | それでも熱いフレームだけ、そのフレームの全ビンを一律に下げる | 同梱の NHVSing ONNX は出力の直前に `[-1, 1]` のクリップを持つ。熱い mel を渡すとそこで波形が潰れ、**後段では戻せない**。抑えるならボコーダに入る前しかない |

どちらも **ln-mel への足し算**です。スペクトルの形は変えません。掛かっていないもの:

* クランプなし（`mel_vmin` / `mel_vmax` は**描画レンジ**であって、出力には掛けていない）
* スムージングなし
* ポストネットなし

**`render_phrase` が返す `mel` は音響モデルの生出力**で、上の 2 つは入っていません。
`wav` のほうは 2 つを掛けた mel から作られています。下のセルで両方を組み立てて、
実際にボコーダへ渡っている mel を見ます。

In [ ]:
from svs.render import (render_phrase, _mel_loudness_offset, _mel_frame_limit,
                        _MEL_FRAME_CLIP_THR)
r = render_phrase(ctx, phr, seed=0)
mel, wav, f0o = r["mel"], r["wav"], r["f0"]        # mel は音響モデルの生出力
T = mel.shape[1]

# クランプしていない証拠。していれば mel_vmin を下回るはずがない。
vmin = ctx.ac_cfg.get("mel_vmin"); vmax = ctx.ac_cfg.get("mel_vmax")
print(f"mel.min() = {mel.min():.3f}   mel_vmin = {vmin}   -> 下回る: {mel.min() < vmin}")
print(f"mel.max() = {mel.max():.3f}   mel_vmax = {vmax}")
print(f"mel_vmin を下回る要素の割合: {(mel < vmin).mean()*100:.2f}%")

# ボコーダへ渡る uv は uv_hard =（F0 モデルの vuv > 0.5）かつ（pau でない）。
frame_ph = np.repeat(np.asarray(r["seq"].phonemes), r["ph_durs"].astype(int))[:T]
uv_hard = ((np.asarray(f0o["uv"])[:T] > 0.5) & (frame_ph != "pau")).astype(np.float32)

delta = _mel_loudness_offset(mel, uv_hard)                 # ① ラウドネス合わせ
fed = _mel_frame_limit(mel + delta, ctx.frame_rate)        # ② フレームリミッタ
print(f"\n① ラウドネス合わせ  delta = {delta:+.3f} ln ({delta*20/np.log(10):+.1f} dB)")

e = np.exp((mel + delta).astype(np.float64)).mean(0)       # フレームごとの mel の大きさ
gain = fed - (mel + delta)
print(f"② フレームリミッタ  閾値 {_MEL_FRAME_CLIP_THR} を超えたフレーム "
      f"{(e > _MEL_FRAME_CLIP_THR).sum()} / {T} ({(e > _MEL_FRAME_CLIP_THR).mean()*100:.2f}%)")
print(f"                    実際に下がったフレーム {(gain.min(0) < -1e-4).sum()} "
      f"({(gain.min(0) < -1e-4).mean()*100:.2f}%)、最大 {gain.min()*20/np.log(10):.1f} dB")
print("                    超過フレームより広いのは、アタック 20ms・リリース 250ms で"
      "下げ量を平滑しているため")

# 出力を再分析した mel は「生 mel」ではなく「渡した mel」と比べる
remel = mel_from_wav(wav, sr=ctx.sample_rate, hop=ctx.hop)
Tc = min(remel.shape[1], T)
print(f"\n再分析 mel との ln-mel MAE : 生 mel 比 {np.abs(remel[:,:Tc]-mel[:,:Tc]).mean():.3f}"
      f"  /  渡した mel 比 {np.abs(remel[:,:Tc]-fed[:,:Tc]).mean():.3f}")

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
plot_mel(mel, fr, ax=axes[0], title="predicted mel (acoustic model, raw)")
plot_mel(fed, fr, ax=axes[1], title=f"mel actually fed to the vocoder (delta={delta:+.2f} ln + limiter)")
plot_mel(remel, fr, ax=axes[2], title="re-analyzed mel from the output audio")
plt.tight_layout(); plt.show()

**入れていないもの**（既知の制限）:

* フレーズ境界のクロスフェード（今は単純加算なので位相不連続が起こりうる）
* `mel_vmin` クランプ（無音部の床を揃える）
* ポストネット（mel を精緻化する小さなネット）
* フレーズごとの音量合わせ（δ はフレーズ単位で求めるが、狙う値は全フレーズ共通の定数）

どれも入れれば音が変わります。入れるなら聴き比べてからにしてください。

**ボコーダは実行ごとに波形が変わります。** ノイズ成分が graph の内部で作られていて、
外から seed を渡せません。同じ mel を 2 回通すと波形で max 0.14（ピーク 0.69 に対して）、
再分析 ln-mel で MAE 0.2 ほど違います。A/B で聴き比べるときは、この分を下回る差は
読み取れないと思ってください。上のセルで波形の一致を検算していないのもこのためです。

## 4. フレーズ分割とタイムライン結合

* **分割**: 休符が `split_silence_sec`（既定 0.5 秒）以上ならフレーズ境界。
  未満なら直前ノートの `gap_after_sec` になり、同じフレーズ内に留まります。
* **配置**: 各フレーズを `start_sec − 先頭pau − 先頭onset` の位置へ置く。
  こうすると**音節核が `start_sec` ちょうど**に来ます。先頭 pau だけ引くと、
  フレーズは自分の先頭子音の長さだけ遅れます（その量はフレーズごとに違うので、
  一律に補正しても直りません）。
* **曲頭のはみ出し**: 引いた結果が負になるフレーズがあれば、`render_score` が
  **曲全体**をその分だけ後ろへずらします。フレーズ個別に 0 でクランプすると
  そのフレーズだけ遅れて相対位置が崩れるためです。
* **結合**: 単純加算。**波形には何もしません**（音量は mel の段で決まっている）。
  旧挙動の「全体のピークを 0.95 へ」は `render_score(..., normalize="peak")` に残してあります。

**注意点**: 単純加算なので、フレーズが重なると境界で位相が不連続になり得ます。
今の楽譜は休符で分かれているため実害は出ていませんが、
フレーズを詰めた譜面を作る場合はクロスフェードを検討してください。

In [ ]:
phrases = load_score(ROOT / "examples" / "kirakiraboshi.ust")
print(f"{len(phrases)} フレーズ\n")
prev_end = 0.0
for i, p in enumerate(phrases):
    s = postprocess_durations(p, rest_before=1e9)
    dur = float(s.dur_sec.sum())
    # render.py と同じ式: 先頭 pau（交渉後の実値）と先頭 onset を引く
    n_on0 = len(split_syllable(p.notes[0].phonemes)[0]) if p.notes else 0
    onset0 = float(s.dur_sec[1:1 + n_on0].sum()) if n_on0 else 0.0
    place = p.start_sec - float(s.dur_sec[0]) - onset0
    print(f"  フレーズ{i}: start={p.start_sec:5.2f}s  配置={place:5.2f}s  長さ={dur:4.2f}s  "
          f"先頭onset={onset0*1000:3.0f}ms  前フレーズとの間隔={place - prev_end:+5.2f}s")
    prev_end = place + dur

## 5. 改造ポイント

| やりたいこと | 触るファイル |
|---|---|
| 歌詞→音素の規則を変える | `dict/<lang>.mora`（`load_score(mora_table=...)`） |
| 音素インベントリ/核を変える | `dict/<lang>.phonemes`（行順が ID）/ `dict/<lang>.yaml` の `nucleus:` |
| 子音・母音の floor を変える | `configs/recipes/<db>.yaml` の `duration:` → `python -m duration_model.build_profile`（`duration_model/profiles/<ckpt名>.json` に焼く） |
| ノート内のグループ分割の規則を変える | `svs/duration_postprocess.py` の `_syllabify` |
| ピッチ補正の強さを変える | `infer.synthesize_f0` の `hann_alpha` / `hann_lead_taper` / `hann_trail_taper` |
| 抑揚・ビブラートの強さ | `plain_scale` / `vib_scale`（→ notebook 03） |
| フレーズ分割の閾値 | `load_score(split_silence_sec=...)` |
| モデルを差し替える | `svs/render.py` の `DEFAULT_*` 定数 |

## 6. 多言語対応の実態

言語ごとに差し替えるものは `dict/<lang>.yaml`（言語パック）に集めてあります:

* 音素インベントリと ID 空間（`dict/<lang>.phonemes`、行順が ID、id 0 は `pau`）
* 核音素（`nucleus:`）、無声化母音の正規化（`vowel_normalize:`）、無音記号（`silence:` / `sil_map:`）
* 楽譜の読み取り（`score:` — UST の文字コード、休符・メリスマ記号、メリスマの既定母音）
* 歌詞→音素（`dict/<lang>.mora`、`--mora_table` / `load_score(mora_table=...)`）

学習・推論・SVS はこれを参照するだけで、言語固有のリテラルはコード側に残していません。
子音の**個数**は固定していないので、子音数の違う音素表を入れても落ちません。
`dict/en.yaml` も同梱していますが、配布ウェイトは日本語 3 話者だけです。


In [ ]:
import duration_model.phoneset as ps
print(f"音素数   : {ps.active().n_phonemes}")
print(f"子音数   : {ps.active().n_consonants}  (個数は固定していない)")
print(f"核音素   : {sorted(ps.active().nuclei)}")
print(f"cl は核か: {is_nucleus_ph('cl')}  (持続長側・F0 側とも子音で統一)")

---

## 次に読むもの

* **01_svs_demo.ipynb** — 実際に歌わせて各段を可視化する
* **03_f0_control.ipynb** — 抑揚・ビブラートの強さを変える（時間変化・スタイルの連続混合）